In [23]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict
import pandas as pd


class Alert(ABC):

    @property
    @abstractmethod
    def id(self) -> int: ...

    @abstractmethod
    def should_raise(self) -> bool: ...

    @abstractmethod
    def describe(self) -> str: ...


@dataclass
class Pathogen:
    org_id: Optional[int]
    org_name: str
    danger_weight: float
    time_window_days: int
    ward_thresholds: Dict[str, int] 
    staff_threshold: int

    def get_ward_threshold(self, ward_size: int) -> int:
        if ward_size <= 5:
            return self.ward_thresholds["5"]
        elif ward_size <= 10:
            return self.ward_thresholds["10"]
        else:
            return self.ward_thresholds["20"]


@dataclass
class MicrobiologyAlert(Alert):
    counter_id: int
    pathogen: Pathogen
    ward_id: int
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    @property
    def org_id(self) -> Optional[int]:
        return self.pathogen.org_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )


@dataclass
class WardAlert(Alert):
    counter_id: int
    ward_id: int
    pathogen: Pathogen
    ward_size: int
    start_time: datetime
    alert_type: str
    curr_patient_number: int

    @property
    def id(self) -> int:
        return self.counter_id

    def should_raise(self) -> bool:
        return self.curr_patient_number >= self.pathogen.get_ward_threshold(self.ward_size)

    def describe(self) -> str:
        thresh = self.pathogen.get_ward_threshold(self.ward_size)
        status = "⚠ ALERT" if self.should_raise() else "OK"
        return (
            f"[{status}][{self.alert_type}] Ward {self.ward_id} "
            f"({self.ward_size} beds) | {self.pathogen.org_name} | "
            f"{self.curr_patient_number}/{thresh} patients @ {self.start_time}"
        )


PATHOGEN_REGISTRY = {
    "CLOSTRIDIUM DIFFICILE": Pathogen(
        org_id=None, org_name="CLOSTRIDIUM DIFFICILE",
        danger_weight=3.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=2,
    ),
    "GRAM NEGATIVE ROD": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD",
        danger_weight=1.5, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "CANDIDA ALBICANS": Pathogen(
        org_id=None, org_name="CANDIDA ALBICANS",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "STREPTOCOCCUS": Pathogen(
        org_id=None, org_name="STREPTOCOCCUS",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=4,
    ),
    "HAEMOPHILUS INFLUENZAE": Pathogen(
        org_id=None, org_name="HAEMOPHILUS INFLUENZAE",
        danger_weight=1.0, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=3,
    ),
    "YEAST unspeciated": Pathogen(
        org_id=None, org_name="YEAST unspeciated",
        danger_weight=1.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 3},
        staff_threshold=4,
    ),
    "ACINETOBACTER BAUMANNII COMPLEX": Pathogen(
        org_id=None, org_name="ACINETOBACTER BAUMANNII COMPLEX",
        danger_weight=3.0, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=2
    ),
    "ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER": Pathogen(
        org_id=None, org_name="ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER", 
        danger_weight=1.5, time_window_days=2,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=2
    ),
    "CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)": Pathogen(
        org_id=None, org_name="CORYNEBACTERIUM SPECIES (DIPHTHEROIDS)",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "ESCHERICHIA COLI": Pathogen(
        org_id=None, org_name="ESCHERICHIA COLI",
        danger_weight=3, time_window_days=3,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "KLEBSIELLA PNEUMONIAE": Pathogen(
        org_id=None, org_name="KLEBSIELLA PNEUMONIAE", 
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 1},
        staff_threshold=1
    ),
    "MYCELIA STERILIA": Pathogen(
        org_id=None, org_name="MYCELIA STERILIA", 
        danger_weight=1, time_window_days=4, 
        ward_thresholds={"5": 2, "10": 3, "20": 4},
        staff_threshold=4
    ),
    "MYCOBACTERIUM AVIUM COMPLEX": Pathogen(
        org_id=None, org_name="MYCOBACTERIUM AVIUM COMPLEX",
        danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 1, "10": 2, "20": 4},
        staff_threshold=3
    ),
    "POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS": Pathogen(
        org_id=None, org_name="POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "PROTEUS MIRABILIS": Pathogen(
        org_id=None, org_name="PROTEUS MIRABILIS",
        danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 2, "10": 3, "20": 4},
        staff_threshold=3
    ),
    "PSEUDOMONAS AERUGINOSA": Pathogen(
        org_id=None, org_name="PSEUDOMONAS AERUGINOSA",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "STAPH AUREUS COAG +": Pathogen(
        org_id=None, org_name="STAPH AUREUS COAG +",
        danger_weight=3, time_window_days=1,
        ward_thresholds={"5": 1, "10": 1, "20": 2},
        staff_threshold=1
    ),
    "STAPHYLOCOCCUS, COAGULASE NEGATIVE": Pathogen(
        org_id=None, org_name="STAPHYLOCOCCUS, COAGULASE NEGATIVE",


danger_weight=1.5, time_window_days=1,
        ward_thresholds={"5": 2, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #2": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #2",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #3": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #3",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
    "GRAM NEGATIVE ROD #4": Pathogen(
        org_id=None, org_name="GRAM NEGATIVE ROD #4",
        danger_weight=2, time_window_days=2,
        ward_thresholds={"5": 1, "10": 2, "20": 3},
        staff_threshold=2
    ),
}


In [ ]:
import sqlite3
import pandas as pd

# Assuming db_path and desired_tables are defined earlier, e.g.:
db_path = ''  # From your MIMIC/OOP project [cite:16] PATH TO
desired_tables = ['PATIENTS', 'CAREGIVERS', 'D_ITEMS', 'ADMISSIONS', 'ICUSTAYS', 'NOTEEVENTS', 'MICROBIOLOGYEVENTS', 'TRANSFERS', 'OUTPUTEVENTS', 'CHARTEVENTS']

from contextlib import contextmanager

@contextmanager
def get_db_connection(db_path):
    conn = sqlite3.connect(db_path)
    try:
        yield conn
    finally:
        conn.close()

with get_db_connection(db_path) as conn:
    db_tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )["name"].str.upper().tolist()

for tbl in desired_tables:
    if tbl in db_tables:
        print(f"Table {tbl} exists.")  # Or your logic here
    else:
        print(f"Table {tbl} missing.")


Table PATIENTS exists.
Table CAREGIVERS exists.
Table D_ITEMS exists.
Table ADMISSIONS exists.
Table ICUSTAYS exists.
Table NOTEEVENTS exists.
Table MICROBIOLOGYEVENTS exists.
Table TRANSFERS exists.
Table OUTPUTEVENTS exists.
Table CHARTEVENTS exists.


In [25]:
# Load tables into data dict (run AFTER table check)
data = {}
with get_db_connection(db_path) as conn:  # Reuse your context manager
    for tbl in desired_tables:
        if tbl in db_tables:
            df_name = tbl.upper()  
            data[df_name] = pd.read_sql_query(f"SELECT * FROM {tbl}", conn)
            print(f"Loaded {df_name} ({len(data[df_name])} rows, {data[df_name].shape[1]} cols)")

# Access like your CSV vars:
patients = data['PATIENTS']
caregivers = data[ 'CAREGIVERS']
d_items = data['D_ITEMS']
admissions = data['ADMISSIONS']
icustays = data['ICUSTAYS']
noteevents = data['NOTEEVENTS']
microbiologyevents = data['MICROBIOLOGYEVENTS']
transfers = data['TRANSFERS']
outputevents = data['OUTPUTEVENTS']
chartevents = data['CHARTEVENTS']


Loaded PATIENTS (100 rows, 7 cols)
Loaded CAREGIVERS (7567 rows, 3 cols)
Loaded D_ITEMS (12487 rows, 9 cols)
Loaded ADMISSIONS (129 rows, 18 cols)
Loaded ICUSTAYS (136 rows, 11 cols)
Loaded NOTEEVENTS (0 rows, 11 cols)
Loaded MICROBIOLOGYEVENTS (2003 rows, 16 cols)
Loaded TRANSFERS (524 rows, 13 cols)
Loaded OUTPUTEVENTS (11319 rows, 13 cols)
Loaded CHARTEVENTS (758274 rows, 15 cols)


In [26]:
chartevents.head()

,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ITEMID,CHARTTIME,STORETIME,CGID,VALUE,VALUENUM,VALUEUOM,WARNING,ERROR,RESULTSTATUS,STOPPED
0,5279021,40124,126179,279554,223761,2130-02-04 04:00:00,2130-02-04 04:35:00,19085,95.9,95.9,?F,0.0,0.0,None,None
1,5279022,40124,126179,279554,224695,2130-02-04 04:25:00,2130-02-04 05:55:00,18999,2222221.7,2222221.7,cmH2O,0.0,0.0,None,None
2,5279023,40124,126179,279554,220210,2130-02-04 04:30:00,2130-02-04 04:43:00,21452,15.0,15.0,insp/min,0.0,0.0,None,None
3,5279024,40124,126179,279554,220045,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,94.0,94.0,bpm,0.0,0.0,None,None
4,5279025,40124,126179,279554,220179,2130-02-04 04:32:00,2130-02-04 04:43:00,21452,163.0,163.0,mmHg,0.0,0.0,None,None


In [27]:
from datetime import timedelta
from collections import defaultdict

microbiologyevents.columns = microbiologyevents.columns.str.upper()
icustays.columns           = icustays.columns.str.upper()

In [28]:
import pandas as pd

WARD_ID_TARGET = 52
WARD_SIZE = 10  # choose 5/10/20 bucket

microbiologyevents["CHARTDATE"] = pd.to_datetime(microbiologyevents["CHARTDATE"], errors="coerce")

ward52_hadm = icustays.loc[
    (icustays["FIRST_WARDID"] == WARD_ID_TARGET) | (icustays["LAST_WARDID"] == WARD_ID_TARGET),
    "HADM_ID"
].dropna().unique()

ward52_pos = microbiologyevents[
    microbiologyevents["HADM_ID"].isin(ward52_hadm)
    & microbiologyevents["ORG_NAME"].notna()
    & microbiologyevents["CHARTDATE"].notna()
].copy()

ward52_pos["ORG_NAME"] = ward52_pos["ORG_NAME"].astype(str).str.upper().str.strip()
ward52_pos = ward52_pos.sort_values(["ORG_NAME", "CHARTDATE"])

In [29]:
ward52_discharge = icustays[(icustays["FIRST_WARDID"] == WARD_ID_TARGET) 
                            & (icustays["LAST_WARDID"] == WARD_ID_TARGET)][["SUBJECT_ID", "OUTTIME"]]
ward52_discharge.head()

,SUBJECT_ID,OUTTIME
0,10076,2107-03-31 06:55:09
3,40310,2144-12-31 21:02:59
4,10104,2120-08-25 15:41:49
8,10006,2164-10-25 12:21:07
9,41976,2201-11-19 16:43:30


In [30]:
ward52_pos = ward52_pos.merge(ward52_discharge, on="SUBJECT_ID", how="left")
ward52_pos

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,ITEMID,SPEC_TYPE_DESC,ORG_ITEMID,ORG_NAME,ISOLATE_NUM,AB_ITEMID,AB_NAME,DILUTION_TEXT,DILUTION_COMPARISON,DILUTION_VALUE,INTERPRETATION,OUTTIME
0,431904,41914,101361,2145-12-06,2145-12-06 16:00:00,70062,SPUTUM,80304.0,ACINETOBACTER BAUMANNII COMPLEX,1.0,90008.0,TRIMETHOPRIM/SULFA,<=1,<=,1.0,S,2145-12-14 18:00:12
1,431905,41914,101361,2145-12-06,2145-12-06 16:00:00,70062,SPUTUM,80304.0,ACINETOBACTER BAUMANNII COMPLEX,2.0,90028.0,CEFEPIME,16,=,16.0,I,2145-12-14 18:00:12
2,431906,41914,101361,2145-12-06,2145-12-06 16:00:00,70062,SPUTUM,80304.0,ACINETOBACTER BAUMANNII COMPLEX,1.0,90028.0,CEFEPIME,16,=,16.0,I,2145-12-14 18:00:12
3,431907,41914,101361,2145-12-06,2145-12-06 16:00:00,70062,SPUTUM,80304.0,ACINETOBACTER BAUMANNII COMPLEX,1.0,90022.0,AMPICILLIN/SULBACTAM,=>32,=>,32.0,R,2145-12-14 18:00:12
4,431908,41914,101361,2145-12-06,2145-12-06 16:00:00,70062,SPUTUM,80304.0,ACINETOBACTER BAUMANNII COMPLEX,1.0,90020.0,IMIPENEM,=>16,=>,16.0,R,2145-12-14 18:00:12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552,136150,10132,197611,2123-08-26,2123-08-26 22:18:00,70079,URINE,80075.0,YEAST,1.0,NaN,None,None,None,NaN,None,2123-08-24 15:11:37
553,136153,10132,197611,2123-08-27,2123-08-27 08:49:00,70062,SPUTUM,80075.0,YEAST,1.0,NaN,None,None,None,NaN,None,2123-08-24 15:11:37
554,426855,40310,186361,2144-07-14,2144-07-14 15:14:00,70062,SPUTUM,80075.0,YEAST,1.0,NaN,None,None,None,NaN,None,2144-12-31 21:02:59
555,426875,40310,186361,2144-07-23,2144-07-23 02:07:00,70062,SPUTUM,80075.0,YEAST,1.0,NaN,None,None,None,NaN,None,2144-12-31 21:02:59


In [31]:
ward52_hadm

array([198503, 157609, 177678, 142345, 155297, 186361, 172454, 195911,
       164869, 188574, 198480, 109698, 114867, 167754, 172082, 171878,
       101361, 174863, 130681, 140372, 124073, 129273, 151798, 175237,
       100375, 176016, 197611, 118192, 142539, 176805, 167612],
      dtype=int64)

In [32]:
ward52_pos["OUTTIME"] = pd.to_datetime(ward52_pos["OUTTIME"])

In [33]:
ward52_pos.groupby("ORG_NAME")["SUBJECT_ID"].nunique().sort_values(ascending=False)

ORG_NAME
STAPHYLOCOCCUS, COAGULASE NEGATIVE                 5
YEAST                                              4
POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS    4
ESCHERICHIA COLI                                   4
STAPH AUREUS COAG +                                2
PSEUDOMONAS AERUGINOSA                             2
PROTEUS MIRABILIS                                  2
KLEBSIELLA PNEUMONIAE                              2
GRAM NEGATIVE ROD #2                               2
CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION       2
GRAM NEGATIVE ROD #4                               1
GRAM NEGATIVE ROD(S)                               1
ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER     1
MYCELIA STERILIA                                   1
MYCOBACTERIUM AVIUM COMPLEX                        1
ORGANISM                                           1
GRAM NEGATIVE ROD #3                               1
PRESUMPTIVE STREPTOCOCCUS BOVIS                    1
CORYNEBACTERIUM SPECIES (DIPHTHEROIDS

In [34]:
ward52_pos.groupby(["ORG_NAME", "CHARTDATE"])["SUBJECT_ID"].nunique().max()

1

In [35]:
episodes_all = []
alert_objects = {}  # episode_id → MicrobiologyAlert

counter_id = 0

for org_name, pathogen in PATHOGEN_REGISTRY.items():
    df = ward52_pos[ward52_pos["ORG_NAME"] == org_name.upper()].copy()
    if df.empty:
        continue
    
    df = df.sort_values("CHARTDATE").reset_index(drop=True)
    
    episode_ids = []
    episode_id = -1
    active_patients = {}

    for _, row in df.iterrows():
        current_date = row["CHARTDATE"]
        
        # Remove patients who have already left
        active_patients = {
            pid: outtime for pid, outtime in active_patients.items()
            if pd.notna(outtime) and outtime >= current_date
        }
        
        if len(active_patients) == 0:
            # New episode
            episode_id += 1
            counter_id += 1
            alert_objects[(org_name, episode_id)] = MicrobiologyAlert(
                counter_id=counter_id,
                pathogen=pathogen,
                ward_id=WARD_ID_TARGET,
                ward_size=WARD_SIZE,
                start_time=current_date,
                alert_type="MICROBIOLOGY",
                curr_patient_number=1,
            )
        else:
            alert = alert_objects[(org_name, episode_id)]
            # Only increment if this is a NEW patient not seen before in this episode
            if row["SUBJECT_ID"] not in active_patients:
                alert.curr_patient_number += 1

        active_patients[row["SUBJECT_ID"]] = row["OUTTIME"]
        episode_ids.append(episode_id)

    df["EPISODE_ID"] = episode_ids
    episodes_all.append(df)

In [36]:
alerts_df = pd.DataFrame([
    {
        "counter_id": alert.counter_id,
        "ward_id": alert.ward_id,
        "org_name": alert.pathogen.org_name,
        "start_time": alert.start_time,
        "curr_patient_number": alert.curr_patient_number,
        "threshold": alert.pathogen.get_ward_threshold(alert.ward_size),
        "should_raise": alert.should_raise(),
        "description": alert.describe(),
    }
    for alert in alert_objects.values()
])

alerts_df.sort_values("start_time")

,counter_id,ward_id,org_name,start_time,curr_patient_number,threshold,should_raise,description
0,1,52,CLOSTRIDIUM DIFFICILE,2105-05-30,1,1,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | CL...
50,51,52,MYCOBACTERIUM AVIUM COMPLEX,2105-05-30,1,2,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | MYCOBAC...
2,3,52,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",2107-03-23,1,1,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | AS...
17,18,52,ESCHERICHIA COLI,2119-10-22,1,1,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | ES...
16,17,52,ESCHERICHIA COLI,2119-10-22,1,1,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | ES...
...,...,...,...,...,...,...,...,...
114,115,52,PROTEUS MIRABILIS,2202-02-15,1,3,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...
116,117,52,PROTEUS MIRABILIS,2202-02-15,1,3,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...
112,113,52,PROTEUS MIRABILIS,2202-02-15,1,3,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...
113,114,52,PROTEUS MIRABILIS,2202-02-15,1,3,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...


In [37]:
# 1. Aggregate episodes_df as before
episodes_df = pd.concat([
    df.groupby("EPISODE_ID").agg(
        start_time=("CHARTDATE", "min"),
        end_time=("OUTTIME", "max"),
        culture_events=("ROW_ID", "count"),
        unique_patients=("SUBJECT_ID", "nunique"),
    ).assign(org_name=df["ORG_NAME"].iloc[0])
    for df in episodes_all
], ignore_index=True)

# 2. Build alerts_df from alert_objects
alerts_df = pd.DataFrame([
    {
        "counter_id": alert.counter_id,
        "ward_id": alert.ward_id,
        "org_name": alert.pathogen.org_name,
        "start_time": alert.start_time,
        "threshold": alert.pathogen.get_ward_threshold(alert.ward_size),
        "ward_size": alert.ward_size,
        "danger_weight": alert.pathogen.danger_weight,
        "should_raise": alert.should_raise(),
        "description": alert.describe(),
        "curr_patient_number": alert.curr_patient_number,
    }
    for alert in alert_objects.values()
])

# 3. Merge on org_name + start_time
combined_df = episodes_df.merge(
    alerts_df,
    on=["org_name", "start_time"],
    how="left"
).sort_values("start_time")

In [38]:
combined_df

,start_time,end_time,culture_events,unique_patients,org_name,counter_id,ward_id,threshold,ward_size,danger_weight,should_raise,description,curr_patient_number
0,2105-05-30,2105-06-11 06:57:03,1,1,CLOSTRIDIUM DIFFICILE,1,52,1,10,3.0,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | CL...,1
934,2105-05-30,2105-06-11 06:57:03,2,1,MYCOBACTERIUM AVIUM COMPLEX,51,52,2,10,1.5,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | MYCOBAC...,1
2,2107-03-23,2107-03-31 06:55:09,1,1,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",3,52,1,10,1.5,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | AS...,1
129,2119-10-22,2119-11-14 17:03:49,2,1,ESCHERICHIA COLI,18,52,1,10,3.0,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | ES...,1
130,2119-10-22,2119-11-14 17:03:49,2,1,ESCHERICHIA COLI,5,52,1,10,3.0,True,[⚠ ALERT][MICROBIOLOGY] Ward 52 (10 beds) | ES...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2561,2202-02-15,2200-11-03 19:00:37,1,1,PROTEUS MIRABILIS,116,52,3,10,1.5,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...,1
2560,2202-02-15,2200-11-03 19:00:37,1,1,PROTEUS MIRABILIS,115,52,3,10,1.5,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...,1
2559,2202-02-15,2200-11-03 19:00:37,1,1,PROTEUS MIRABILIS,114,52,3,10,1.5,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...,1
2570,2202-02-15,2200-03-28 17:15:09,1,1,PROTEUS MIRABILIS,115,52,3,10,1.5,False,[OK][MICROBIOLOGY] Ward 52 (10 beds) | PROTEUS...,1


In [39]:
episodes_all = []

for org_name, pathogen in PATHOGEN_REGISTRY.items():
    df = ward52_pos[ward52_pos["ORG_NAME"] == org_name.upper()].copy()
    if df.empty:
        continue

    df = df.sort_values("CHARTDATE").reset_index(drop=True)

    # Assign episode IDs using fixed window from episode start
    episode_ids = []
    episode_id = 0
    episode_start = None

    for _, row in df.iterrows():
        if episode_start is None:
            episode_start = row["CHARTDATE"]
            episode_end = row["OUTTIME"]
            episode_ids.append(episode_id)
        else:
            # Is any previous patient from this episode still on the ward?
            still_occupied = pd.notna(episode_end) and row["CHARTDATE"] <= episode_end
            gap = (row["CHARTDATE"] - episode_start).total_seconds() / 86400

            if still_occupied or gap <= pathogen.time_window_days:
                # Join existing episode — a previous patient is still present
                # OR we're within the time window
                if pd.notna(row["OUTTIME"]):
                    if pd.notna(episode_end):
                        episode_end = max(episode_end, row["OUTTIME"])
                    else:
                        episode_end = row["OUTTIME"]
            else:
                # No overlap, beyond window → new episode
                episode_id += 1
                episode_start = row["CHARTDATE"]
                episode_end = row["OUTTIME"]

            episode_ids.append(episode_id)

    df["EPISODE_ID"] = episode_ids

    # Aggregate episode stats
    ep = df.groupby("EPISODE_ID").agg(
        start_time=("CHARTDATE", "min"),
        end_time=("OUTTIME", "max"),
        culture_events=("ROW_ID", "count"),
        unique_patients=("SUBJECT_ID", "nunique"),
    ).reset_index(drop=True)

    ep["org_name"] = pathogen.org_name
    ep["window_days"] = pathogen.time_window_days
    ep["threshold"] = pathogen.get_ward_threshold(WARD_SIZE)
    ep["alert"] = ep["unique_patients"] >= ep["threshold"]

    episodes_all.append(ep)


In [40]:
episodes_df = pd.concat(episodes_all, ignore_index=True) if episodes_all else pd.DataFrame()

In [41]:
if episodes_df.empty:
    print("No pathogen episodes found in ward 52 for the registry organisms.")
else:
    print("Episodes (counters) found:", len(episodes_df))
    print("Alerts fired (episodes where unique_patients >= threshold):", int(episodes_df["alert"].sum()))

    print("\nAlerts per pathogen:")
    print(
        episodes_df[episodes_df["alert"]]
        .groupby("org_name")
        .agg(
            alert_episodes=("alert", "size"),
            first_alert=("start_time", "min"),
            last_alert=("end_time", "max"),
            max_patients=("unique_patients", "max"),
            threshold=("threshold", "first"),
            window_days=("window_days", "first"),
        )
        .sort_values("alert_episodes", ascending=False)
        .to_string()
    )

    print("\nAll ALERT episodes (each row = one counter instance):")
    print(
        episodes_df[episodes_df["alert"]]
        .sort_values(["org_name", "start_time"])
        .to_string(index=False)
    )


Episodes (counters) found: 31
Alerts fired (episodes where unique_patients >= threshold): 17

Alerts per pathogen:
                                                 alert_episodes first_alert          last_alert  max_patients  threshold  window_days
org_name                                                                                                                             
ESCHERICHIA COLI                                              4  2119-10-22 2169-05-12 11:35:14             1          1            3
POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS               4  2144-07-14 2202-02-17 03:10:56             1          1            1
KLEBSIELLA PNEUMONIAE                                         2  2145-12-06 2202-02-17 03:10:56             1          1            1
PSEUDOMONAS AERUGINOSA                                        2  2145-12-02 2202-02-17 03:10:56             1          1            1
STAPH AUREUS COAG +                                           2  2144-07-23 2145-

In [42]:
episodes_df

,start_time,end_time,culture_events,unique_patients,org_name,window_days,threshold,alert
0,2105-05-30,2105-06-11 06:57:03,1,1,CLOSTRIDIUM DIFFICILE,3,1,True
1,2145-12-06,2145-12-14 18:00:12,25,1,ACINETOBACTER BAUMANNII COMPLEX,3,1,True
2,2107-03-23,2107-03-31 06:55:09,1,1,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",2,1,True
3,2169-05-07,2169-05-12 11:35:14,1,1,CORYNEBACTERIUM SPECIES (DIPHTHEROIDS),2,2,False
4,2119-10-22,2119-11-14 17:03:49,28,1,ESCHERICHIA COLI,3,1,True
5,2120-08-25,2120-08-25 15:41:49,16,1,ESCHERICHIA COLI,3,1,True
6,2155-12-16,2155-12-17 12:58:21,14,1,ESCHERICHIA COLI,3,1,True
7,2169-05-07,2169-05-12 11:35:14,14,1,ESCHERICHIA COLI,3,1,True
8,2145-12-06,2145-12-14 18:00:12,12,1,KLEBSIELLA PNEUMONIAE,1,1,True
9,2201-08-11,2202-02-17 03:10:56,65,1,KLEBSIELLA PNEUMONIAE,1,1,True


In [43]:
with get_db_connection(db_path) as conn:
    cursor = conn.cursor()
    
    # Create table if it doesn't exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS MICROALERTS (
            ALERT_ID INTEGER PRIMARY KEY,
            WARD_ID INTEGER NOT NULL,
            ORG_ID INTEGER,
            ORG_NAME TEXT NOT NULL,
            NUM_PATIENTS INTEGER NOT NULL,
            CULTURE_EVENTS INTEGER NOT NULL,
            START_TIME TIMESTAMP NOT NULL,
            END_TIME TIMESTAMP NOT NULL,
            SEVERITY INTEGER NOT NULL,
            THRESHOLD INTEGER NOT NULL,
            WINDOW_DAYS INTEGER NOT NULL
        )
    ''')
    conn.commit()
    
    # Clear existing records for this ward
    cursor.execute('DELETE FROM MICROALERTS WHERE WARD_ID = ?', (WARD_ID_TARGET,))
    conn.commit()
    
    # Insert alert records from episodes_df
    counter_id = 0
    for idx, row in episodes_df[episodes_df["alert"]].iterrows():
        pathogen = PATHOGEN_REGISTRY[row["org_name"]]
        
        # Convert timestamps to strings
        start_time_str = str(row["start_time"])
        end_time_str = str(row["end_time"])

        cursor.execute(
            '''
            INSERT INTO MICROALERTS 
            (ALERT_ID, WARD_ID, ORG_ID, ORG_NAME, NUM_PATIENTS, CULTURE_EVENTS, 
             START_TIME, END_TIME, SEVERITY, THRESHOLD, WINDOW_DAYS)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''',
            (
                int(counter_id),
                int(WARD_ID_TARGET),
                int(pathogen.org_id) if pathogen.org_id is not None else None,
                str(row["org_name"]),
                int(row["unique_patients"]),
                int(row["culture_events"]),
                start_time_str,
                end_time_str,
                int(pathogen.danger_weight * 10),
                int(row["threshold"]),
                int(row["window_days"]),
            )
        )
        counter_id += 1
    
    conn.commit()

    # Verify count
    alert_count = cursor.execute(
        'SELECT COUNT(*) FROM MICROALERTS WHERE WARD_ID = ?',
        (WARD_ID_TARGET,)
    ).fetchone()[0]

    # Verify data
    alerts_verify = pd.read_sql_query(
        'SELECT * FROM MICROALERTS WHERE WARD_ID = ? ORDER BY START_TIME',
        conn,
        params=(WARD_ID_TARGET,)
    )

print(f"Created/updated MICROALERTS table with {alert_count} alert records for Ward {WARD_ID_TARGET}")
print(f"\nTotal alerts in database: {len(alerts_verify)}")
print(alerts_verify.head(10))

Created/updated MICROALERTS table with 17 alert records for Ward 52

Total alerts in database: 17
   ALERT_ID  WARD_ID ORG_ID                                         ORG_NAME  \
0         0       52   None                            CLOSTRIDIUM DIFFICILE   
1         2       52   None   ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER   
2         3       52   None                                 ESCHERICHIA COLI   
3         4       52   None                                 ESCHERICHIA COLI   
4         9       52   None  POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS   
5        15       52   None                              STAPH AUREUS COAG +   
6        13       52   None                           PSEUDOMONAS AERUGINOSA   
7        16       52   None                              STAPH AUREUS COAG +   
8         1       52   None                  ACINETOBACTER BAUMANNII COMPLEX   
9         7       52   None                            KLEBSIELLA PNEUMONIAE   

   NUM_PATIENTS  CULT

In [44]:
# build a lookup of organism name → org_itemid (int)
microbiologyevents["ORG_NAME"] = (
    microbiologyevents["ORG_NAME"].astype(str)
    .str.upper()
    .str.strip()
)

org_id_map = (
    microbiologyevents
    .dropna(subset=["ORG_NAME", "ORG_ITEMID"])
    .groupby("ORG_NAME")["ORG_ITEMID"]
    .first()          # grab first occurrence for each name
    .astype(int)      # convert float → int
    .to_dict()
)

print(f"{len(org_id_map)} organism names have an ORG_ITEMID; sample:\n",
      list(org_id_map.items())[:8])

# write the ids back into MICROALERTS
with get_db_connection(db_path) as conn:
    cur = conn.cursor()
    for org_name, oid in org_id_map.items():
        cur.execute(
            "UPDATE MICROALERTS SET ORG_ID = ? WHERE ORG_NAME = ?",
            (oid, org_name)
        )
    conn.commit()

    # refresh the dataframe we use for checking
    alerts_verify = pd.read_sql_query(
        'SELECT * FROM MICROALERTS WHERE WARD_ID = ? ORDER BY START_TIME',
        conn,
        params=(WARD_ID_TARGET,)
    )

# optional: keep a local copy named alerts_table too
alerts_table = alerts_verify.copy()

print("after update, alerts table head")
alerts_verify.head()

46 organism names have an ORG_ITEMID; sample:
 [('ACINETOBACTER BAUMANNII', 80194), ('ACINETOBACTER BAUMANNII COMPLEX', 80304), ('ALPHA STREPTOCOCCI', 80052), ('ANAEROBIC GRAM POSITIVE ROD(S)', 80079), ('ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER', 80239), ('BACTEROIDES FRAGILIS GROUP', 80112), ('BETA STREPTOCOCCUS GROUP B', 80045), ('CANDIDA ALBICANS, PRESUMPTIVE IDENTIFICATION', 80254)]
after update, alerts table head


,ALERT_ID,WARD_ID,ORG_ID,ORG_NAME,NUM_PATIENTS,CULTURE_EVENTS,START_TIME,END_TIME,SEVERITY,THRESHOLD,WINDOW_DAYS,STATUS
0,0,52,80139,CLOSTRIDIUM DIFFICILE,1,1,2105-05-30 00:00:00,2105-06-11 06:57:03,30,1,3,OPEN
1,2,52,80239,"ASPERGILLUS SP. NOT FUMIGATUS, FLAVUS OR NIGER",1,1,2107-03-23 00:00:00,2107-03-31 06:55:09,15,1,2,OPEN
2,3,52,80002,ESCHERICHIA COLI,1,28,2119-10-22 00:00:00,2119-11-14 17:03:49,30,1,3,OPEN
3,4,52,80002,ESCHERICHIA COLI,1,16,2120-08-25 00:00:00,2120-08-25 15:41:49,30,1,3,OPEN
4,9,52,80293,POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS,1,1,2144-07-14 00:00:00,2144-12-31 21:02:59,30,1,1,OPEN
